# Lab 3 - Model Comparison
## 0. Setup and Data Loading
Setting random seed to 414 and loading data.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 414

df = pd.read_csv('data/jolpica_results_2018_2024.csv')
df.head()

## 1. Feature Engineering and Framing
Target: `is_points_finish` (1 if position <= 10, else 0).

In [ ]:
# Convert position to numeric, handling potential coercions
df['position_num'] = pd.to_numeric(df['position'], errors='coerce').fillna(99)
df['is_points_finish'] = (df['position_num'] <= 10).astype(int)
print(df['is_points_finish'].value_counts(normalize=True))

## 2. Temporal Splitting
Train: 2018-2023
Test: 2024

In [ ]:
train_df = df[df['season'] < 2024].copy()
test_df = df[df['season'] == 2024].copy()

features = ['grid', 'circuit_id', 'constructor_id']
target = 'is_points_finish'

# Handle missing grids by imputing with median
train_df['grid'] = pd.to_numeric(train_df['grid'], errors='coerce').fillna(20)
test_df['grid'] = pd.to_numeric(test_df['grid'], errors='coerce').fillna(20)

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

## 3. Preprocessing Pipeline


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['grid']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['circuit_id', 'constructor_id'])
    ])


## 4. Model Training & Evaluation

In [ ]:
results = []

def evaluate_model(name, y_train_pred, y_test_pred, features_used, why):
    train_f1 = f1_score(y_train, y_train_pred, average='macro')
    test_f1 = f1_score(y_test, y_test_pred, average='macro')
    results.append({
        'Model': name,
        'Features': features_used,
        'Validation': '2024 Season',
        'Train Macro F1': round(train_f1, 4),
        'Test Macro F1': round(test_f1, 4),
        'Train-Test Gap': round(train_f1 - test_f1, 4),
        'WHY this result': why
    })

# Baseline 1: Majority Class (Predict 0)
evaluate_model(
    'Baseline (Majority)', 
    np.zeros(len(y_train)), 
    np.zeros(len(y_test)), 
    '-', 
    'Always predicts \'No Points\'; fails to identify any scoring drivers, resulting in high recall for class 0 but 0 for class 1.'
)

# Baseline 2: Domain Heuristic (Top 10 Grid => Points)
train_heuristic = (X_train['grid'] <= 10).astype(int)
test_heuristic = (X_test['grid'] <= 10).astype(int)
evaluate_model(
    'Baseline (Domain Heuristic)', 
    train_heuristic, 
    test_heuristic, 
    'grid <= 10', 
    'Strong heuristic since F1 grid position largely determines final finish, but blind to race dynamics, pitstops and DNF.'
)


In [ ]:
# Model 1: Logistic Regression
lr_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression(random_state=RANDOM_SEED, max_iter=1000))
])
lr_pipeline.fit(X_train, y_train)
evaluate_model(
    'Logistic Regression', 
    lr_pipeline.predict(X_train), 
    lr_pipeline.predict(X_test), 
    'grid, circuit_id, constructor_id',
    'Good at capturing the strong linear link between starting P1-P5 and scoring, but lacks flexibility to model interactions.'
)

# Model 2: Ridge Classifier
ridge_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', RidgeClassifier(random_state=RANDOM_SEED))
])
ridge_pipeline.fit(X_train, y_train)
evaluate_model(
    'Ridge Classifier', 
    ridge_pipeline.predict(X_train), 
    ridge_pipeline.predict(X_test), 
    'grid, circuit_id, constructor_id',
    'Very similar to logistic regression but uses L2 regularization on target; fails to capture complex team-track interactions.'
)

# Model 3: Random Forest
rf_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=100, max_depth=5))
])
rf_pipeline.fit(X_train, y_train)
evaluate_model(
    'Random Forest', 
    rf_pipeline.predict(X_train), 
    rf_pipeline.predict(X_test), 
    'grid, circuit_id, constructor_id',
    'Better at identifying nonlinear team strengths (e.g. Red Bull starting P15 still scoring). The Train-Test gap points to slight overfitting.'
)


## 5. Comparison Table

In [ ]:
results_df = pd.DataFrame(results)
results_df

In [ ]:
# Export Table to markdown for Lab
print(results_df.to_markdown(index=False))